# Burgers' equation
$$
u_t + uu_x = 0.1u_{xx}
$$


In [1]:
import sys
import os
sys.path.append(os.path.abspath(r'...\TT-IRLS'))
import scikit_tt as scikit
import numpy as np
import scipy.io as sio
import tensor_auxiliary as aux


In [2]:
data = sio.loadmat(r'...\burgers_data.mat')
u = np.real(data['u_out'])
x = np.real(data['x'][0])
t = np.real(data['t'][0])
dt = np.real(data['dt'][0][0])
dx = x[2]-x[1]
print(u.shape)
u = u.T

(101, 256)


In [3]:
n, m = u.shape
ut = np.zeros((n,m))
for i in range(n):
    ut[i,:] = aux.FiniteDiff(u[i ,:],dt,1)
ux = np.zeros((n,m))
for i in range(m):
    ux[:,i] = aux.FiniteDiff(u[:,i],dx,1)
uxx = np.zeros((n,m))
for i in range(m):
    uxx[:,i] = aux.FiniteDiff(u[:,i],dx,2)
uxxx = np.zeros((n,m))
for i in range(m):
    uxxx[:,i] = aux.FiniteDiff(u[:,i],dx,3)


In [4]:
T1 = 0
T2 = 10
M1 = 64
M2 = 192

In [5]:
U = np.array([u[M1:M2,T1:T2].reshape((T2-T1)*(M2-M1)),
              ux[M1:M2,T1:T2].reshape((T2-T1)*(M2-M1)),
             uxx[M1:M2,T1:T2].reshape((T2-T1)*(M2-M1))])

v = np.array([ut[M1:M2,T1:T2].reshape((T2-T1)*(M2-M1))])
print(v.shape)
print(U.shape)

P = [lambda t: 1, lambda t: t ,lambda t:t**2]
p = len(P)
print(v.shape)

(1, 1280)
(3, 1280)
(1, 1280)


In [6]:
p = len(P)

core_type_1 = np.zeros([1, p, 1, 1])
core_type_1[0, 0, 0, 0] = 1

core_type_2 = np.zeros([1, p, 1, 1])
core_type_2[0, 1, 0, 0] = 1

core_type_4 = np.zeros([1, 1, 1, 1])
core_type_4[0, 0, 0, 0] = 1

cores = [core_type_2]
cores.append(core_type_2)
cores.append(core_type_1)
cores.append(-1*core_type_4)
coefficient_tensor = scikit.TT(cores) # u ux

cores = [core_type_1]
cores.append(core_type_1)
cores.append(core_type_2)
cores.append(0.1*core_type_4)
coefficient_tensor += scikit.TT(cores) # uxx

xi_exact = coefficient_tensor
xi_exact_num = xi_exact.full().flatten()
# print(xi_exact_num)

In [7]:
xi = aux.mandy_cm(U, v, P, threshold=1e-8)
xi_num1 = xi.full().flatten()
xi_formatted = [f"{x:.4f}" for x in xi_num1]
print(", ".join(xi_formatted))

-0.0000, 0.1011, -0.0043, 0.0004, -0.0036, 0.0008, -0.0127, -0.0406, 0.0006, -0.0068, 0.0490, 0.0344, -0.9998, 0.0030, -0.0014, 0.0042, 0.0363, -0.0016, 0.0050, -0.0245, -0.0164, -0.0005, -0.0016, 0.0003, 0.0057, -0.0061, 0.0000


In [8]:
iter=500
d, m = U.shape  
p = len(P)  
n = p ** d  
#b0 = xi_num1
b0=range(n)
e=1e-5
for i in range(iter):
    psi=aux.build_psi(U,P,lam=4e-2,beta=b0,eps=1e-5)
    xi=aux.coefficient_solving(U,psi,P,v,1e-8)
    b1=xi.full().flatten()
    if abs(b1-b0).all()<e:
        break
    b0=b1
print(i)
xi_num2 = xi.full().flatten()
xi_formatted = [f"{x:.4f}" for x in xi_num2]
print(", ".join(xi_formatted))

488
0.0000, 0.0999, 0.0000, -0.0001, -0.0018, -0.0000, -0.0000, 0.0004, 0.0000, 0.0000, -0.0000, -0.0000, -0.9999, -0.0000, -0.0000, -0.0000, 0.0002, -0.0000, 0.0000, -0.0008, -0.0004, -0.0000, -0.0001, -0.0000, -0.0000, 0.0001, -0.0000


In [9]:
rel_errors = np.linalg.norm(xi_num1 - xi_exact_num) / np.linalg.norm(xi_exact_num)
print("OLS",rel_errors)
rel_errors = np.linalg.norm(xi_num2 - xi_exact_num) / np.linalg.norm(xi_exact_num)
print("IRLS",rel_errors)


OLS 0.08787311912526047
IRLS 0.0020412558366924276


In [10]:
candidates = []
for u in ['', 'u', 'u2']:
    for ux in ['', 'ux', 'ux2']:
        for uxx in ['', 'uxx', 'uxx2']:
            candidates.append(u + ux + uxx )
candidates[0] = '1'

In [11]:
idx = [i for i,val in enumerate(xi_exact_num) if val != 0]
res = [f"{xi_num2[i]}{candidates[i]}" for i in idx]
print(res)

['0.09992954232626156uxx', '-0.9999273167381543uux']
